# Chapter 3 &mdash; Independent Component Analysis 
## 3.1. Rationale

Independent Component Analysis (ICA) is a mathematical technique to decompose a signal into a set of factors that respect the following constraints:

- They are statistically independent of one another &mdash; that is, observing one does not influence the likelihood of observing the other
- The original signal can be reconstructed as a linear combination of said factors

Another constraint (which must be mentioned for completeness, but is less important for a first grasp of ICA) is that the probability of observing any given value of the factors should follow a non-Gaussian distribution. Interested students can refer to [Makeig et al. (1996)](https://proceedings.neurips.cc/paper_files/paper/1995/file/754dda4b1ba34c6fa89716b85d68532b-Paper.pdf) or [Shlens (2014)](https://doi.org/10.48550/arXiv.1404.2986) for a deeper treatment of ICA mathematics and related assumptions. Otherwise, the book ["Independent Component Analysis: A Tutorial Introduction" by James V. Stone (MIT Press)](https://ieeexplore.ieee.org/servlet/opac?bknumber=6267342) is available from [BUR &mdash; Rovereto University Library](https://www.biblioteca.unitn.it/en/bur-rovereto-university-library).

ICA has been used for EEG analyses since the mid 1990s ([Makeig et al., 1996](https://proceedings.neurips.cc/paper_files/paper/1995/file/754dda4b1ba34c6fa89716b85d68532b-Paper.pdf)), with the goal of isolating brain from non-brain signals &mdash; in particular, eye and muscle noise. 

Typically, EEG researchers use ICA in a **five-step** process:

1. Fit an ICA model to the data (that is, run code that factors the signal into independent components)
2. Visualize the resulting components with topoplots and voltage time series
    - Topoplots are colour-coded scalp voltage maps projected in 2D
3. Evaluate the plots to seek signs of non-brain activity 
    - This is the hardest part: experience and data quality are vital
4. Factor out (that is, exclude) non-brain components 
5. Reconstruct the data using only brain components 

Applications of ICA to neuroimaging data assume that brain signals are a linear combination of independent sources. In other words, they assume that the following idea holds true:

$$S^{(t)} = \alpha_1x^{(t)}_1 + \alpha_2x^{(t)}_2 + \alpha_3x^{(t)}_3 + ... + \alpha_nx^{(t)}_n$$

where:

- $S^{(t)}$ is the brain signal at time $t$
- $x^{(t)}_1, x^{(t)}_2, x^{(t)}_3, ..., x^{(t)}_n$ are $n$ independent factors, measured at time $t$. _Independent_ means that the value of any factor is the same regardless of the value of the others 
- $\alpha_1, \alpha_2, \alpha_3, ..., \alpha_n$ are the _weights_ of the other factors, which represent how much each factor contributes to the final signal at time $t$

In the case of EEG, the factors can only be brain areas, eyes, muscles, and other noise sources. Therefore, the equation above translates into something like:

$$ EEG^{(t)} = \alpha_1\text{Area}^{(t)}_1 + \alpha_2\text{Area}^{(t)}_2 + \alpha_3\text{Eyeblinks}^{(t)} + \alpha_4\text{Heartbeat}^{(t)} + ... + \alpha_n\text{Muscle Contraction}^{(t)}$$

If you think about it, the independence assumption is deeply flawed, as different brain and body parts cannot be thought of as separate entities. 

In particular, it is true that: 

- Brain areas interact with each other
- Heartbeats modulate brain activity, and vice versa
- Muscle contractions can originate from the brain

Therefore, ICA shall be used with great care: while it can factor brain signals into _statistically_ independent components, those components do not necessarily make neuroscientific sense.

Perhaps the only EEG component that can be considered approximately independent from the others is the eyeblink: while it does originate from the brain, it has three characteristics that make it reasonable to factor out. Those are:

1. **Lack of connection with cognitive dynamics:** eyeblinks are usually an act of no cognitive significance, which we perform merely to protect our eyes from drying out 
2. **Relative amplitude:** the amplitude of eyeblink artefacts is much larger than the amplitude of brain signals, so they must be removed because they alter the signal significantly
3. **Lack of better methods:** currenly, the main alternative to factoring out eyeblinks with ICA is to discard the signal segments that contain the eyeblinks. If the eyeblinks are close to interesting brain responses, this can cause an undesirable loss of data

## 3.2. Fitting

This is the first step of the ICA lifecycle, and its very heart: here, the EEG signal is factored into statistically independent components that can be recombined back into the signal.

You have seen before that EEG data are represented as a matrix with dimension `channels x time`. The set of statistically independent components found by ICA will also be a matrix, with as many rows as there are components and as many columns as there are time points (`components x time`).

The transformation of the EEG data matrix into a components matrix is done by [multiplication](https://learninglab.rmit.edu.au/maths-statistics/linear-algebra/matrices-getting-started/m3-matrix-multiplication/) with an _unmixing matrix_. The unmixing matrix cannot be _any_ matrix: it must be such that the resulting components are statistically independent of one another. 

The reverse transformation from components to EEG data is also done by matrix multiplication &mdash; specifically, by multiplying the matrix of components with another one called _mixing matrix_. 

<p align="center">
<img src="files/ica-nutshell.png" width="1000"/>
</p>

The process of fitting ICA consists mainly in finding the _unmixing_ matrix: once that is found, the _mixing_ matrix can be derived from it as the one that inverts the unmixing. \
Simplifying for educational purposes, we can think about this as finding some coefficient $\alpha$, whose inverse will necessarily be $\dfrac{1}{\alpha}$:

$$ y = \alpha \cdot x \implies x = \dfrac{1}{\alpha} \cdot y $$

### 3.2.1. Beware of randomness

Fitting an ICA is an iterative process where candidate matrices are tried and rejected until the right one is found &mdash; where _"right"_ means _"such that the resulting components are statistically independent of one another"_. Conceptually, this process articulates over the following steps:

1. Perform unmixing 
2. Measure the statistical independence of the resulting components
3. Adjust the the unmixing matrix to increase the statistical independence of components
4. Repeat steps 1-3 until reaching one of the following conditions:
    - Statistical independence is maximised
    - A predefined number of iterations has been reached

Basically, fitting an ICA is a sophisticated form of trial-and-error, where the fitting algorithm performs an ideal walk across the space of all possible unmixing matrices until it finds the right one. \
Importantly, the starting point of this walk is not always the same, as the first trial (Step 1 in the list above) is performed with a matrix of [pseudorandom](https://en.wikipedia.org/wiki/Pseudorandomness) values. Therefore, applying ICA twice to the same dataset might yield two different sets of components due to different values of the initial matrix, which result in different walks accross the space of solutions. The likelihood of encountering this issue increases if the fitting process is set to end upon reaching a predefined number of iterations rather than a true solution, as different paths might end in significantly different places after the same number of steps.     

To increase the reproducibility of ICA &mdash; that is, to avoid obtaining two different solutions from two different ICA runs &mdash; one can set a _random seed_. A random seed (also called _random state_) is a number that forms the basis for the generation of a set of pseudorandom numbers. By setting a random seed, one ensures that the starting point of the walk across the space of all possible unmixing matrices is always the same, therefore ensuring that the fitting process is reproducible. 

--- 

❗ Note that setting a random seed ensures reproducibility _within_ a given computer, but does not ensure reproducibility _between_ different computers. In other words, all of us obtained consistent results from multiple ICA runs, but some of us obtained different decompositions than the others. The most likely explanation for this phenomenon is that different computers might find different solutions to the same problem due to one or more of the following causes:

- Different [pseudorandom number generators](https://en.wikipedia.org/wiki/Pseudorandom_number_generator): different computers might use the random seed in different ways, thereby setting different starting points for ICA's walk despite using the same seed
- Different decimal approximations: different computers might represent decimal numbers to different degrees of precision, resulting in small differences that propagate along the fitting process until yielding significantly different results 

Once again, these issues can be particularly severe if the fitting process is set to end upon reaching a predefined number of iterations rather than a true solution.

 ### 3.2.2. Good Practices for Computational Reproducibility

The problem of obtaining consistent ICA decompositions across computers is an example of a computational reproducibility problem.

Computational reproducibility is the ability to reproduce the exact results of a computational procedure (for example, a computerised data-analysis pipeline). While it might seem trivial, computational reproducibility is a known issue in science and technology: to give a rough sense of its relevance, searching it on Google Scholar returns [more than one million results](https://scholar.google.com/scholar?hl=it&as_sdt=0%2C5&q=computational+reproducibility&btnG=) as of late 2025.  

Computational reproducibility is influenced by a myriad factors, and determining the exact causes behind a reproducibility failure can be very difficult &mdash; especially in the absence of detailed information about all the software and hardware involved in and around a given process. However, close adherence to a few good practices can increase the reproducibility of any process.

In the case of ICA, the good practices for computational reproducibility are:

- Set a random seed
- Ensure that the fitting process stops upon reaching a true solution. To this end, it is useful to:
    - Increase the maximum number of iterations, to make sure that the fitting does not stop too early (and therefore, too distant from a true solution)
    - Decrease the tolerance parameter. The tolerance parameter is a threshold for change between unmixing matrices: when the difference between two unmixing matrices from two consecutive attempts is less than the tolerance parameter, the fitting can be considered concluded because the fact that changes are getting very small suggests that the true solution is very close

Clearly, ICA fitting cannot go on forever, so one might be forced to make less iterations and/or keep the tolerance parameter to a relatively high value. In these cases like in all others, it is very important to document which values were used for the parameters, so readers of your work can relate your output to the context that generated it. 

ICA-specific practices must always add to more general-purpose reproducibility practices, like the consistency of software versions: if two users use different versions of a toolbox (like MNE-Python), they might use slightly different implementations of the same algorithm and therefore, inject differences between their workflows. The importance of consistency applies to a given toolbox as well as its dependencies, and that is why we all use the same `conda` environment in this course. 

If you want to know more about computational reproducibility, you can read section 5 of [Good Programming Practices](./docs/good-programming-practices.md).

### 3.2.3. The actual code

The code below initializes an `ICA` object, which is available from the `preprocessing` module of the `mne` package. 

The ICA object is initialised with a random state of 0, which means that the initial values of the unmixing matrix will be generated by some complex operation applied on number 0 (see [Wikipedia](https://en.wikipedia.org/wiki/Pseudorandom_number_generator) for details on pseudorandom number generators). \
Beside the random seed, it is possible to set important parameters like the number of desired components (`n_components`, which is equal to the number of channels by default) or the maximum number of iterations (`max_iter`, which is `1000` by default). The complete list of possible parameters (and their description) is available from the official MNE-Python documentation at [this](https://mne.tools/stable/generated/mne.preprocessing.ICA.html#mne.preprocessing.ICA) link.

After initialising the ICA object and assigning it to variable `ica`, the code calls method `fit()` and gives it the post-filtering epochs as input. This basically means that the post-filtering epochs are concatenated along the temporal axis to yield a single EEG data matrix of shape `channels x time`, which is used to find the _unmixing_ matrix and the corresponding _mixing_ matrix. Note that this will not return the decomposed data: it will merely find the ICA solution, without applying it to the data yet.   

In [ ]:
ica = mne.preprocessing.ICA(random_state=0)
ica.fit(post_rejection_epochs)

The process of fitting ICA involves large matrix multiplications, which must be repeated for a number of iterations. This can be very long and time-consuming, depending on the computational power at hand and the size of the data. 

Luckily, it is possible to save the result of the fitting process and write it to a file, so a given dataset can be decomposed once and the result can be used forever without repeating the fitting process. This also opens the door to sharing the result of your ICA decomposition with colleagues, students and readers of your work.   

The following code does just that: it saves the fitted ICA to a file named `fitted_ica.fif` within directory `data`. The argument `overwrite=True` allows you to rewrite the existing file should you need to re-run the procedure.

In [ ]:
ica.save(fname="data/fitted_ica.fif", overwrite=True)

Pre-computed ICA solutions can then be read from file and assigned to a Python variable, as follows:

In [ ]:
ica = mne.preprocessing.read_ica(fname="data/fitted_ica.fif")

## **3.3. Components Selection**

Once the data are factored into a set of components, we can plot those components in a variety of formats. The aim is to evaluate their appearance and classify them as cerebral activity versus muscular, ocular, cardiac, or environmental noise.

One way to visualize components is to plot their scalp topography &mdash; that is, a colour-coded map of how component intensity distributes over the scalp. Usually, strongly positive activity values are red in a scalp topography, while strongly positive values are blue and intermediate values have intermediate colours like shades of orange, yellow and green.

Scalp topographies are usually accompanied by a series of supporting plots, including:

- The component's power spectrum
- The component's event-related potential (ERP) &mdash; that is, its average across epochs 
- The component's ERP image &mdash; that is, its ERP accompanied by a colour-coded representation of all single trials

All these plots are relevant to evaluate a component, as they all provide complementary information that, taken together, can build a complete picture.

We are especially interested in finding and rejecting ocular components, which are usually revealed by the following set of information:

- Intense activity near the eyes. In particular, it is usually true that:

    - A unipolar concentration of activity around the sagittal midline suggests _vertical_ eye movements
    - A left-right bipolar concentration of activity suggests _horizontal_ eye movements
    
- Highest power at lower frequencies (that is, 5 Hz and below)

- Distribution that is random over trials, or stimulus-locked in case of stimuli that can elicit eye movements (e.g., TMS pulses can elicit blinks)

The code below calls method `plot_components()` on the `ica` object, using the post-filtering epochs as data of interest and picking only the first 30 components (in order of variance explained) for brevity. This will generate one scalp topography for each component. Clicking once on the scalp topography will open a new panel that displays the topography alongside all other plots &mdash; including the power spectrum, the ERP and the ERP image.

In [ ]:
ica.plot_components(inst=post_rejection_epochs, picks=range(30))

A second (and complementary) method to evaluate independent components is `plot_sources()`, which plots the time series of components. Visualizing time series is useful because it gives a sense of how components behave in time. For example, eye components are mostly stationary except for brief, large-amplitude spikes (for blinks) or step curves of varying length (for horizontal movements). 

In [ ]:
ica.plot_sources(inst=post_rejection_epochs)

Note that calling `plot_components()` or `plot_sources()` will visualize the result of the ICA decomposition, but only for visualization purposes: the aim is only for users to _inspect_ components and decide which ones are worth rejecting. To actually reject them, you need to assign a list of component numbers to the `exclude` attribute of the `ica` object. For example, running `ica.exclude = [0,1,2]` will reject components number 0, 1, and 2 (that is, those who explain the most, second-most, and third-most variance).

The code in the following cell assigns a list containing a single value (`[0]`) to `ica.exclude`, meaning that the `ica` object is attributed a list containing the index of a component that needs rejection (`[0]`). Subsequently, the component is rejected using method `apply()`, then epochs are averaged and plotted for visualization as usual. 

In [ ]:
ica.exclude = [0]
ica.apply(post_rejection_epochs.load_data())
post_ica_epochs = post_rejection_epochs
post_ica_tep = post_ica_epochs.average()
post_ica_tep.plot();